In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
   StructType, StructField, StringType, DoubleType
)
from datetime import datetime, timezone
import uuid
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]
NORMAL_DRIFT_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_drift_normal"
)
PERFORMANCE_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_model_performance"
)
REGISTRY_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_model_run_registry"
)
EVENTS_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_monitoring_events"
)
normal_drift_df = spark.table(NORMAL_DRIFT_TABLE)
performance_df = spark.table(PERFORMANCE_TABLE)
registry_df = spark.table(REGISTRY_TABLE)
pipeline_run_id = str(uuid.uuid4())
event_time = datetime.now(timezone.utc).isoformat()
latest_model_run_id = (
   registry_df
   .orderBy(F.desc("registered_at_utc"))
   .first()["run_id"]
)
print("Monitoring execution:", pipeline_run_id)
print("Model run:", latest_model_run_id)

Monitoring execution: f4bdaf28-1b5b-49e9-966a-38b77b86259c
Model run: b3ca314766bc4afbb27d9bb58180dfbf


In [0]:
# Drift thresholds
PSI_THRESHOLD = 0.20
KS_PVALUE_THRESHOLD = 0.05
# Initial model-performance thresholds
MIN_RECALL = 0.70
MIN_AVERAGE_PRECISION = 0.50
print("PSI alert threshold:", PSI_THRESHOLD)
print("KS p-value threshold:", KS_PVALUE_THRESHOLD)
print("Minimum recall:", MIN_RECALL)
print("Minimum average precision:", MIN_AVERAGE_PRECISION)

PSI alert threshold: 0.2
KS p-value threshold: 0.05
Minimum recall: 0.7
Minimum average precision: 0.5


In [0]:
drift_pdf = normal_drift_df.toPandas()

events = []

for _, row in drift_pdf.iterrows():

    psi = row["psi"]

    ks_pvalue = row["ks_pvalue"]

    reasons = []

    if pd.notna(psi) and psi > PSI_THRESHOLD:

        reasons.append(

            f"PSI {psi:.4f} exceeds {PSI_THRESHOLD}"

        )

    if (

        pd.notna(ks_pvalue)

        and ks_pvalue < KS_PVALUE_THRESHOLD

    ):

        reasons.append(

            f"KS p-value {ks_pvalue:.6g} is below "

            f"{KS_PVALUE_THRESHOLD}"

        )

    is_alert = len(reasons) > 0

    events.append({

        "pipeline_run_id": pipeline_run_id,

        "event_time_utc": event_time,

        "model_run_id": latest_model_run_id,

        "check_name": f"drift:{row['feature']}",

        "status": "ALERT" if is_alert else "OK",

        "severity": "WARNING" if is_alert else "INFO",

        "metric_name": "psi",

        "observed_value": (

            float(psi) if pd.notna(psi) else None

        ),

        "threshold_value": float(PSI_THRESHOLD),

        "message": (

            "; ".join(reasons)

            if is_alert

            else f"No configured drift threshold exceeded for {row['feature']}"

        ),

        "source_table": NORMAL_DRIFT_TABLE

    })

print("Feature checks evaluated:", len(drift_pdf))

print("Drift alerts:", sum(

    event["status"] == "ALERT" for event in events

))
 

Feature checks evaluated: 30
Drift alerts: 2


In [0]:
performance_pdf = performance_df.toPandas()
# Use the latest measurement for each metric.
performance_pdf = (
   performance_pdf
   .sort_values("measured_at_utc")
   .groupby("metric_name", as_index=False)
   .tail(1)
)
performance_thresholds = {
   "recall": MIN_RECALL,
   "average_precision": MIN_AVERAGE_PRECISION
}
for _, row in performance_pdf.iterrows():
   metric_name = row["metric_name"]
   value = float(row["metric_value"])
   # Always record the metric; alert only for configured metrics.
   threshold = performance_thresholds.get(metric_name)
   is_alert = (
       threshold is not None and value < threshold
   )
   events.append({
       "pipeline_run_id": pipeline_run_id,
       "event_time_utc": event_time,
       "model_run_id": str(row["model_run_id"]),
       "check_name": f"performance:{metric_name}",
       "status": "ALERT" if is_alert else "OK",
       "severity": "WARNING" if is_alert else "INFO",
       "metric_name": metric_name,
       "observed_value": value,
       "threshold_value": (
           float(threshold) if threshold is not None else None
       ),
       "message": (
           f"{metric_name} {value:.4f} is below "
           f"the minimum {threshold:.4f}"
           if is_alert
           else (
               f"{metric_name} meets the configured threshold"
               if threshold is not None
               else f"{metric_name} recorded; no alert threshold configured"
           )
       ),
       "source_table": PERFORMANCE_TABLE
   })
print("Performance checks evaluated:", len(performance_pdf))
print("Total alerts so far:", sum(
   event["status"] == "ALERT" for event in events
))

Performance checks evaluated: 5
Total alerts so far: 2


In [0]:
event_schema = StructType([
   StructField("pipeline_run_id", StringType(), False),
   StructField("event_time_utc", StringType(), False),
   StructField("model_run_id", StringType(), False),
   StructField("check_name", StringType(), False),
   StructField("status", StringType(), False),
   StructField("severity", StringType(), False),
   StructField("metric_name", StringType(), False),
   StructField("observed_value", DoubleType(), True),
   StructField("threshold_value", DoubleType(), True),
   StructField("message", StringType(), False),
   StructField("source_table", StringType(), False)
])
events_df = spark.createDataFrame(events, schema=event_schema)
(events_df.write
   .format("delta")
   .mode("append")
   .saveAsTable(EVENTS_TABLE))
print("Monitoring history saved to:", EVENTS_TABLE)

Monitoring history saved to: workspace.default.fraud_monitoring_events


In [0]:
saved_events = spark.table(EVENTS_TABLE).filter(
   F.col("pipeline_run_id") == pipeline_run_id
)
display(
   saved_events
   .orderBy(
       F.when(F.col("status") == "ALERT", 0).otherwise(1),
       F.col("check_name")
   )
)
print("Checks:", saved_events.count())
print(
   "Alerts:",
   saved_events.filter(F.col("status") == "ALERT").count()
)
display(
   saved_events
   .groupBy("status", "severity")
   .count()
)

pipeline_run_id,event_time_utc,model_run_id,check_name,status,severity,metric_name,observed_value,threshold_value,message,source_table
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V12,ALERT,WARNING,psi,4.2917669373032045E-4,0.2,KS p-value 0.014438 is below 0.05,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V16,ALERT,WARNING,psi,1.9834646686405156E-4,0.2,KS p-value 0.0338155 is below 0.05,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:Amount,OK,INFO,psi,9.744413922173699E-5,0.2,No configured drift threshold exceeded for Amount,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:Time,OK,INFO,psi,1.5153576336223877E-4,0.2,No configured drift threshold exceeded for Time,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V1,OK,INFO,psi,1.0657483962135284E-4,0.2,No configured drift threshold exceeded for V1,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V10,OK,INFO,psi,2.1088264789987732E-4,0.2,No configured drift threshold exceeded for V10,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V11,OK,INFO,psi,2.1824510479591635E-4,0.2,No configured drift threshold exceeded for V11,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V13,OK,INFO,psi,1.4431579177657356E-4,0.2,No configured drift threshold exceeded for V13,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V14,OK,INFO,psi,6.970403080813261E-5,0.2,No configured drift threshold exceeded for V14,workspace.default.fraud_drift_normal
f4bdaf28-1b5b-49e9-966a-38b77b86259c,2026-10-11T10:22:16.333495+00:00,b3ca314766bc4afbb27d9bb58180dfbf,drift:V15,OK,INFO,psi,6.329089047403549E-5,0.2,No configured drift threshold exceeded for V15,workspace.default.fraud_drift_normal


Checks: 35
Alerts: 2


status,severity,count
OK,INFO,33
ALERT,WARNING,2


In [0]:
SIMULATED_DRIFT_TABLE = (
   f"{catalog_name}.{schema_name}.fraud_drift_simulated"
)
simulated_df = spark.table(SIMULATED_DRIFT_TABLE)
display(
   simulated_df
   .filter(
       (F.col("feature").isin("Amount", "V14"))
   )
   .select(
       "feature",
       "psi",
       "ks_pvalue",
       "psi_alert",
       "ks_alert"
   )
)
print(
   "Simulated PSI alerts:",
   simulated_df.filter(F.col("psi_alert") == True).count()
)

feature,psi,ks_pvalue,psi_alert,ks_alert
V14,null,0.0,false,true
Amount,null,0.0,false,true


Simulated PSI alerts: 0
